# LRU Cache

A **least-recently-used cache**: fixed capacity, **O(1)** `get` and `put`, and when it's full it evicts the entry untouched the longest. It's the canonical "combine two structures" exercise — a **hash map** for O(1) lookup plus a **doubly linked list** for O(1) recency ordering — and it's exactly how CPython builds `functools.lru_cache` and `OrderedDict` (the linked-lists notebook flagged this).

**Contents**

1. **What it is**
2. **From scratch** — hash map + doubly linked list
3. **The easy way** — `OrderedDict`
4. **The decorator** — `functools.lru_cache`
5. **When to use & cheat-sheet**

## 1. What it is

An **LRU cache** bounds memory by keeping only the K most-recently-used entries. Two operations, both **O(1)**:

- `get(key)` — return the value and mark the key **most recently used**;
- `put(key, value)` — insert/update, mark most recently used, and if over capacity **evict the least recently used**.

No single built-in does both halves: you need **fast lookup by key** (a hash map) *and* **fast reordering by recency** (to know what to evict). The trick is to combine them — a `dict` mapping keys to **nodes of a doubly linked list** ordered by recency. The dict gives O(1) find; the list gives O(1) move-to-front and evict-from-back. (This is the structure behind `functools.lru_cache` — linked-lists and recursion notebooks.)

## 2. From scratch — hash map + doubly linked list

Keep a doubly linked list with **sentinel** head and tail nodes (so there are no edge cases at the ends). The node right after `head` is the **most** recently used; the node right before `tail` is the **least**. On every access, unlink the node and re-add it at the front; on overflow, drop the node before `tail`:

In [1]:
class Node:
    __slots__ = ("key", "value", "prev", "next")
    def __init__(self, key=None, value=None):
        self.key, self.value = key, value
        self.prev = self.next = None

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.map = {}                                # key -> Node, for O(1) lookup
        self.head, self.tail = Node(), Node()        # sentinels: head <-> ... <-> tail
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node):                         # unlink a node, O(1)
        node.prev.next, node.next.prev = node.next, node.prev

    def _add_front(self, node):                      # insert right after head = most recent
        node.next, node.prev = self.head.next, self.head
        self.head.next.prev = self.head.next = node

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node); self._add_front(node)    # touch -> most recently used
        return node.value

    def put(self, key, value):
        if key in self.map:                          # update + promote
            node = self.map[key]
            node.value = value
            self._remove(node); self._add_front(node)
            return
        if len(self.map) >= self.cap:                # full -> evict the LRU (before tail)
            lru = self.tail.prev
            self._remove(lru)
            del self.map[lru.key]
        node = Node(key, value)
        self.map[key] = node
        self._add_front(node)

cache = LRUCache(2)
cache.put(1, 1); cache.put(2, 2)
print("get(1):", cache.get(1))      # 1   (1 is now MRU, 2 is LRU)
cache.put(3, 3)                     # capacity 2 -> evict key 2
print("get(2):", cache.get(2))      # -1  (evicted)
cache.put(4, 4)                     # evict key 1
print("get(1):", cache.get(1), "| get(3):", cache.get(3), "| get(4):", cache.get(4))   # -1 3 4


get(1): 1
get(2): -1
get(1): -1 | get(3): 3 | get(4): 4


The `_add_front` / `_remove` pointer surgery is exactly the doubly-linked-list splicing from the linked-lists notebook — the `dict` just lets you jump to any node in O(1) so you can splice it. Neither structure alone gives O(1) for *both* lookup and recency; together they do.

## 3. The easy way — `OrderedDict`

You rarely hand-roll the linked list, because `collections.OrderedDict` *is* a dict backed by a doubly linked list (linked-lists notebook §6). `move_to_end` promotes a key and `popitem(last=False)` evicts the oldest — both O(1) — so the whole cache is a few lines:

In [2]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.od = OrderedDict()                      # recency order, O(1) reorder

    def get(self, key):
        if key not in self.od:
            return -1
        self.od.move_to_end(key)                     # mark most recently used
        return self.od[key]

    def put(self, key, value):
        if key in self.od:
            self.od.move_to_end(key)
        self.od[key] = value
        if len(self.od) > self.cap:
            self.od.popitem(last=False)              # evict the least recently used (front)

cache = LRUCache(2)
cache.put(1, 1); cache.put(2, 2); cache.get(1); cache.put(3, 3)   # evicts key 2
print("get(2):", cache.get(2), "| get(1):", cache.get(1), "| get(3):", cache.get(3))   # -1 1 3


get(2): -1 | get(1): 1 | get(3): 3


## 4. The decorator — `functools.lru_cache`

For caching **function results** (not a general key-value store) you don't write a class at all — `functools.lru_cache` is an LRU cache wired into a decorator. It's the memoization tool from the recursion notebook, and internally it's the same dict + **circular** doubly linked list:

In [3]:
from functools import lru_cache

@lru_cache(maxsize=2)                # keep results for the 2 most-recently-used arg sets
def square(x):
    print(f"  (computing {x})")
    return x * x

square(2); square(3)                 # both computed
square(2)                            # a hit - returned from cache, no recompute
square(4)                            # evicts 3 (the least recently used)
print("cache_info:", square.cache_info())


  (computing 2)
  (computing 3)
  (computing 4)
cache_info: CacheInfo(hits=1, misses=3, maxsize=2, currsize=2)


`maxsize=None` (or the bare `functools.cache`) makes it an **unbounded** memo with no eviction — the version used for Fibonacci in the recursion and DP notebooks. Bound it with `maxsize=N` whenever the key space is large and you want to cap memory.

## 5. When to use & cheat-sheet

| You need… | Use |
|---|---|
| cache function results, bounded | `functools.lru_cache(maxsize=N)` |
| cache function results, unbounded | `functools.cache` |
| a general bounded key→value cache | `OrderedDict` (`move_to_end` + `popitem`) |
| to understand / customize eviction | hand-rolled `dict` + DLL |
| a different policy (LFU, TTL, …) | a custom structure, or `cachetools` |

**The takeaway:** an LRU cache is the textbook example of **composing structures** — neither a hash map nor a linked list alone gives O(1) for *both* lookup and recency, but together they do. In Python you almost always reach for `lru_cache` or `OrderedDict`; the from-scratch version is for understanding the machinery (and the interview classic "implement an LRU cache").

| Operation | dict + DLL | `OrderedDict` | `lru_cache` |
|---|:---:|:---:|:---:|
| `get` | O(1) | O(1) | O(1) |
| `put` / insert | O(1) | O(1) | O(1) |
| evict LRU | O(1) | O(1) | O(1) (automatic) |
| memory | O(capacity) | O(capacity) | O(maxsize) |